# HVAC v19s retrain — direct upload (no Drive, no OTP)

Uses the **940 MB** bundle `hvac_v19s_tiled_colab.zip` (10,000 tiles, ALL air-device tiles kept — this is the one meant to BEAT v10, unlike the small v19xs that lost air-device recall).

### Steps
1. `Runtime -> Change runtime type -> GPU` (T4 is fine), then run cells 1-2.
2. Click the **folder icon** on the LEFT sidebar (file browser).
3. **Drag `hvac_v19s_tiled_colab.zip` into that panel** (uploads to /content). 940 MB — expect ~5-20 min depending on your upload speed. Wait for the spinner to finish before the next cell.
4. Run cell 3 (it finds the zip; if you skipped the drag it pops a file picker).
5. Run cells 4-5. Training is ~60-90 min on a T4. At the end it downloads `hvac_yolov8s_v19s.pt` -> save it to `C:\...\hvac-takeoff-tool-master\` and tell Claude; the local gate (`gate_track_a.py`) decides ship/keep vs v10 (bar = held-out F1 0.889).

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'NONE -> Runtime > Change runtime type > GPU, then Run all again')

In [ ]:
!pip -q install ultralytics==8.4.51

In [ ]:
# Drag the zip into the left file panel first (recommended), then run this.
from google.colab import files
import zipfile, os, glob
found = glob.glob('/content/*.zip')
if not found:
    up = files.upload()              # picker fallback if you didn't drag it in
    found = ['/content/' + k for k in up]
zname = found[0]
print('using:', zname, f'({os.path.getsize(zname)/1e6:.0f} MB)')
os.makedirs('/content/hvac', exist_ok=True)
with zipfile.ZipFile(zname) as z:
    z.extractall('/content/hvac')
os.chdir('/content/hvac')
print('ready:', os.getcwd(), sorted(os.listdir('.'))[:8])

In [ ]:
# Train. ~60-90 min on a T4 for 10k tiles. (Drop --epochs to 40 for a faster first pass.)
# run-name v19s so the promoted weight is models/hvac_yolov8s_v19s.pt (won't clobber the v19xs run).
!python train_v11.py --data-yaml yolo_dataset_v19s_tiled/data.yaml \
    --epochs 60 --device 0 --batch 16 --imgsz 640 --run-name v19s

In [ ]:
import glob
from google.colab import files
cands = glob.glob('models/hvac_yolov8s_v19s.pt') + glob.glob('runs/**/weights/best.pt', recursive=True)
assert cands, 'No weights found - check the training cell output.'
print('downloading', cands[0])
files.download(cands[0])